In [ ]:
# --- Deep learning framework ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import Adam
from einops import rearrange

# --- Data processing ---
import os
import numpy as np
import pandas as pd
from PIL import ImageFile, Image
import anndata

# --- Scientific computing ---
from scipy.spatial import distance
import scipy.stats as st
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

# --- Spatial transcriptomics ---
import scanpy as sc
import scprep as scp

# --- Pretrained models ---
from transformers import AutoImageProcessor, AutoModel

# --- Custom ViT module ---
from finetune.vision_transformer import vit_small

In [ ]:
# --- Configuration ---
fold = 5                 # LOO-CV fold index (which sample was held out)
tag = '-test'
learning_rate = 1e-5
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## 1. Model Components

Same architecture as `5_VISTA_model_training.ipynb`. Redefined here for standalone inference.
Components: ConvNeXtBlock → GatedFusion → ImprovedMultiScaleConvNeXt → ViT → MultiHeadGAT → GlobalToLocalCrossAttention → AttentionLayer → ImgtoGene.

In [ ]:
# --- ConvNeXtBlock ---
class ConvNeXtBlock(nn.Module):
    """ConvNeXt basic block: 7×7 depthwise conv → LayerNorm → SwiGLU MLP → LayerScale → residual."""
    def __init__(self, dim):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        shortcut = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        return x + shortcut


# --- GatedFusion ---
class GatedFusion(nn.Module):
    """Dynamically weights three branches (3×3 / 5×5 / 7×7) via global pooling + FC + softmax."""
    def __init__(self, dim):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(dim * 3, dim // 2), nn.ReLU(inplace=True),
            nn.Linear(dim // 2, 3), nn.Softmax(dim=1))

    def forward(self, x1, x2, x3):
        b, c, _, _ = x1.size()
        cat_feat = torch.cat([self.avg_pool(x1), self.avg_pool(x2), self.avg_pool(x3)], dim=1).view(b, -1)
        weights = self.fc(cat_feat).view(b, 3, 1, 1, 1)
        return x1 * weights[:, 0] + x2 * weights[:, 1] + x3 * weights[:, 2]


# --- ImprovedMultiScaleConvNeXt ---
class ImprovedMultiScaleConvNeXt(nn.Module):
    """Multi-scale feature extractor: 3×3 / 5×5 / 7×7 branches → gated fusion."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=7, padding=1),
            ConvNeXtBlock(out_channels))
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=5, stride=7, padding=2),
            ConvNeXtBlock(out_channels))
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=7, stride=7, padding=3),
            ConvNeXtBlock(out_channels))
        self.fusion = GatedFusion(out_channels)
        self.post_conv = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.GELU())

    def forward(self, x):
        out = self.fusion(self.branch1(x), self.branch2(x), self.branch3(x))
        return self.post_conv(out)


# --- ViT (FFN-only Transformer) ---
class ConvNeXt_FFN_Transformer(nn.Module):
    """FFN-only transformer — replaces self-attention with feed-forward layers."""
    def __init__(self, channel=32, dim=1024, depth1=2, depth2=4, dropout=0.1):
        super().__init__()
        self.convnext_layers = nn.Sequential(
            *[ConvNeXtBlock(dim=channel) for _ in range(depth1)])
        self.down = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(channel, dim), nn.LayerNorm(dim))
        self.ffn_layers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(dim), nn.Linear(dim, dim * 2), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(dim * 2, dim), nn.Dropout(dropout))
            for _ in range(depth2)])

    def forward(self, x, ct):
        x = self.convnext_layers(x)
        x = self.down(x)
        g = x.unsqueeze(0) + ct
        for ffn in self.ffn_layers:
            g = g + ffn(g)
        return g.squeeze(0)


class ViT(nn.Module):
    """ViT wrapper with input dropout."""
    def __init__(self, channel=32, dim=1024, depth1=2, depth2=4, dropout=0.):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.model = ConvNeXt_FFN_Transformer(
            channel=channel, dim=dim, depth1=depth1, depth2=depth2, dropout=dropout)

    def forward(self, x, ct):
        return self.model(self.dropout(x), ct)


# --- Multi-Head Graph Attention Network (GAT) ---
class GraphAttentionLayer(nn.Module):
    """Single-head GAT layer (Veličković et al., 2018)."""
    def __init__(self, in_features, out_features, dropout=0.2, alpha=0.01, concat=True):
        super(GraphAttentionLayer, self).__init__()
        self.dropout = dropout
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.concat = concat
        self.W = nn.Parameter(torch.empty(size=(in_features, out_features)))
        nn.init.xavier_uniform_(self.W.data, gain=1.414)
        self.a = nn.Parameter(torch.empty(size=(2 * out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=1.414)
        self.leakyrelu = nn.LeakyReLU(self.alpha)

    def forward(self, h, adj):
        Wh = torch.mm(h, self.W)
        e = self._prepare_attentional_mechanism_input(Wh)
        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=1)
        attention = F.dropout(attention, self.dropout, training=self.training)
        h_prime = torch.matmul(attention, Wh)
        return F.elu(h_prime) if self.concat else h_prime

    def _prepare_attentional_mechanism_input(self, Wh):
        Wh1 = torch.matmul(Wh, self.a[:self.out_features, :])
        Wh2 = torch.matmul(Wh, self.a[self.out_features:, :])
        return self.leakyrelu(Wh1 + Wh2.T)


class MultiHeadGAT(nn.Module):
    """Multi-head GAT: first layer concatenates heads, second layer averages."""
    def __init__(self, in_features, nhid, out_features, dropout, alpha, heads=4):
        super(MultiHeadGAT, self).__init__()
        self.dropout = dropout
        self.attentions = [
            GraphAttentionLayer(in_features, nhid, dropout=dropout, alpha=alpha, concat=True)
            for _ in range(heads)]
        for i, attention in enumerate(self.attentions):
            self.add_module('attention_{}'.format(i), attention)
        self.out_att = GraphAttentionLayer(
            nhid * heads, out_features, dropout=dropout, alpha=alpha, concat=False)

    def forward(self, x, adj):
        x = F.dropout(x, self.dropout, training=self.training)
        x = torch.cat([att(x, adj).squeeze(0) for att in self.attentions], dim=1)
        x = F.dropout(x, self.dropout, training=self.training)
        return F.elu(self.out_att(x, adj))


# --- AttentionLayer ---
class AttentionLayer(nn.Module):
    """Learned attention-weighted fusion of two embedding modalities."""
    def __init__(self, in_feat, out_feat, dropout=0.0, act=F.relu):
        super(AttentionLayer, self).__init__()
        self.in_feat = in_feat
        self.out_feat = out_feat
        self.w_omega = nn.Parameter(torch.FloatTensor(in_feat, out_feat))
        self.u_omega = nn.Parameter(torch.FloatTensor(out_feat, 1))
        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.w_omega)
        torch.nn.init.xavier_uniform_(self.u_omega)

    def forward(self, emb1, emb2):
        emb = []
        emb.append(torch.unsqueeze(torch.squeeze(emb1), dim=1))
        emb.append(torch.unsqueeze(torch.squeeze(emb2), dim=1))
        self.emb = torch.cat(emb, dim=1)
        self.v = F.tanh(torch.matmul(self.emb, self.w_omega))
        self.vu = torch.matmul(self.v, self.u_omega)
        self.alpha = F.softmax(torch.squeeze(self.vu) + 1e-6, dim=1)
        a = torch.transpose(self.emb, 1, 2)
        b = torch.unsqueeze(self.alpha, -1)
        emb_combined = torch.matmul(a, b)
        return torch.squeeze(emb_combined)


# --- GlobalToLocalCrossAttention ---
class GlobalToLocalCrossAttention(nn.Module):
    """Cross-attention: spot features (Q) attend to global slide embedding (K,V)."""
    def __init__(self, dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads,
                                           dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, spot_feat, global_feat):
        if global_feat.dim() == 2:
            global_feat = global_feat.unsqueeze(1)
        out, _ = self.attn(query=spot_feat, key=global_feat, value=global_feat)
        return self.norm(spot_feat + out)


## 2. VISTA Main Model (ImgtoGene)

In [ ]:
class ImgtoGene(nn.Module):
    """
    VISTA: Vision-Spatial-Transcriptomic Analysis.
    Pipeline: MultiScaleConvNeXt → ViT + pos embed → GAT (img) → TITAN global
           → GAT (UNI2) → Cross-Attn → Fusion → MLP → gene expression.
    """
    def __init__(self, n_genes=1000):
        super(ImgtoGene, self).__init__()
        self.CBAMConv = ImprovedMultiScaleConvNeXt(in_channels=3, out_channels=32)
        self.x_embed = nn.Embedding(64, 1024)
        self.y_embed = nn.Embedding(64, 1024)
        self.vit = ViT()
        self.gat = MultiHeadGAT(in_features=1024, nhid=1024, out_features=512,
                                heads=8, dropout=0.2, alpha=0.01)
        self.gat1 = MultiHeadGAT(in_features=512, nhid=1024, out_features=512,
                                 heads=8, dropout=0.2, alpha=0.01)
        self.cross_attn = GlobalToLocalCrossAttention(dim=512, num_heads=8)
        self.AttentionAggregationLayer = AttentionLayer(512, 512)
        self.cemb_proj = nn.Sequential(
            nn.Linear(1536, 512), nn.SiLU(), nn.Linear(512, 512))
        self.cemb_proj2 = nn.Sequential(
            nn.Linear(1536, 512), nn.SiLU(), nn.Linear(512, 768))
        self.cemb_proj3 = nn.Sequential(
            nn.Linear(768, 512), nn.SiLU(), nn.Linear(512, 512))
        self.mlp = nn.Sequential(
            nn.Linear(512, 512 * 4), nn.SiLU(), nn.LayerNorm(512 * 4),
            nn.Dropout(0.2), nn.Linear(512 * 4, 512), nn.SiLU(),
            nn.LayerNorm(512), nn.Dropout(0.1), nn.Linear(512, n_genes))
        self.init_weights()

    def init_weights(self):
        """Initialize linear layers with truncated normal distribution."""
        for module in [self.mlp, self.cemb_proj, self.cemb_proj2, self.cemb_proj3]:
            for layer in module:
                if isinstance(layer, nn.Linear):
                    size = layer.weight.size()
                    fan_out, fan_in = size[0], size[1]
                    std = np.sqrt(2.0 / (fan_in + fan_out))
                    layer.weight.data.normal_(0.0, std)
                    layer.bias.data.normal_(0.0, 0.001)

    def forward(self, patches, features, adj, center, positions):
        B, N, C, H, W = patches.shape
        patches = patches.reshape(B * N, C, H, W)

        patches = self.CBAMConv(patches)

        positions = positions.long()
        centers_x = self.x_embed(positions[:, :, 0]).permute(1, 0, 2).squeeze(1)
        centers_y = self.y_embed(positions[:, :, 1]).permute(1, 0, 2).squeeze(1)
        ct = centers_x.unsqueeze(0) + centers_y.unsqueeze(0)

        x = self.vit(patches, ct)
        cemb_f1 = self.gat(x, adj)

        B, N, F = features.shape
        features = features.reshape(B * N, F)
        cemb = self.cemb_proj(features)

        H0_features = self.cemb_proj2(features)
        center = center.squeeze(0).long()
        patch_size_lv0 = 112
        H0_features = H0_features.unsqueeze(0)
        with torch.autocast('cuda', torch.float16), torch.inference_mode():
            slide_embedding2 = titan.encode_slide_from_patch_features(
                H0_features, center, patch_size_lv0)
        global_feat = slide_embedding2.clone().float()
        global_feat = self.cemb_proj3(global_feat).squeeze()

        cemb1 = self.gat1(cemb, adj)
        cemb_ca = self.cross_attn(cemb1, global_feat.unsqueeze(0))
        cemb_f3 = self.AttentionAggregationLayer(
            cemb_f1.squeeze(0), cemb_ca.squeeze(0))

        output = self.mlp(cemb_f3)
        return output.squeeze()

## 3. Dataset and DataLoader

In [ ]:
Image.MAX_IMAGE_PIXELS = 100000000

class Her2ST(torch.utils.data.Dataset):
    """her2st spatial transcriptomics dataset with leave-one-out CV split."""

    def __init__(self, train=True, fold=0):
        super(Her2ST, self).__init__()
        self.exp_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-cnts'
        self.img_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-imgs'
        self.pos_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-spotfiles'
        self.feature_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-UNI2-features'
        self.patch_dir = r'./对比方法/THItoGene-main/data/her2st/data/ST-patches'
        self.gene_list = list(np.load(
            r'./对比方法/THItoGene-main/data/her_hvg_cut_1000.npy', allow_pickle=True))
        self.r = 56

        names = [file for file in os.listdir(self.exp_dir) if file.endswith('.tsv')]
        names.sort()
        names = [i[:2] for i in names]
        names = [item for item in names if item not in ['A1', 'H1', 'H2', 'H3']]
        self.train = train

        te_names = [names[fold]]
        tr_names = list(set(names) - set(te_names))
        self.names = tr_names if train else te_names

        print('Loading imgs...')
        self.img_dict = {i: torch.Tensor(np.array(self.get_img(i))) for i in self.names}
        print('Loading expdata...')
        self.meta_dict = {i: self.get_meta(i) for i in self.names}
        self.feature_dict = {i: self.get_features(i) for i in self.names}
        self.patch_dict = {i: self.get_patches(i) for i in self.names}
        self.exp_dict = {key: self.preprocess(data) for key, data in self.meta_dict.items()}
        self.adj_dict = {
            key: self.construct_knn_image_graph(data, img_feat, k=4, min_degree=2)
            for (key, data), (_, img_feat) in zip(self.exp_dict.items(), self.feature_dict.items())
        }

    def __getitem__(self, index):
        name = self.names[index]
        exps = self.exp_dict[name].X
        features = self.feature_dict[name]
        centers = (np.floor(self.exp_dict[name].obs[['pixel_x', 'pixel_y']]).values).astype(int)
        positions = (np.floor(self.exp_dict[name].obs[['x', 'y']]).values).astype(int)
        patches = self.patch_dict[name]
        adj = self.adj_dict[name]
        return patches, features, torch.Tensor(exps), adj, torch.Tensor(centers), torch.Tensor(positions)

    def __len__(self):
        return len(self.exp_dict)

    def preprocess(self, adata):
        """Filter HVGs, normalize to 10K counts, log1p."""
        adata = adata[:, self.gene_list]
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        return adata

    def get_img(self, name):
        pre = self.img_dir + '/' + name[0] + '/' + name
        fig_name = [file for file in os.listdir(pre) if file.endswith('.jpg')][0]
        path = pre + '/' + fig_name
        print(path)
        return Image.open(path)

    def get_meta(self, name):
        exp_path = self.exp_dir + '/' + name + '.tsv'
        adata = sc.read(exp_path, delimiter='\t')
        pos_path = self.pos_dir + '/' + name + '_selection.tsv'
        df = pd.read_csv(pos_path, sep='\t')
        x = np.around(df['x'].values).astype(int)
        y = np.around(df['y'].values).astype(int)
        id = [str(x[i]) + 'x' + str(y[i]) for i in range(len(x))]
        df.index = id
        adata.obs = adata.obs.merge(df, how='left', left_index=True, right_index=True)
        return adata

    def get_features(self, name):
        return torch.load(self.feature_dir + '/' + name + '.pt')

    def get_patches(self, name):
        return torch.load(self.patch_dir + '/' + name + '.pt')

    def construct_knn_image_graph(self, adata, img_feat, k=4, min_degree=2, eps=1e-6):
        """Build weighted adjacency graph: spatial kNN × image feature cosine similarity."""
        pos = adata.obs[['pixel_x', 'pixel_y']].values.astype(float)
        N = pos.shape[0]
        nbrs = NearestNeighbors(n_neighbors=k + 1).fit(pos)
        _, indices = nbrs.kneighbors(pos)
        knn_idx = indices[:, 1:]
        adj_spatial = np.zeros((N, N), dtype=np.float32)
        for i in range(N):
            adj_spatial[i, knn_idx[i]] = 1.0
        adj_spatial = np.maximum(adj_spatial, adj_spatial.T)
        sim = np.clip(cosine_similarity(img_feat), 0.0, 1.0)
        adj = adj_spatial * sim
        for i in range(N):
            current_degree = int((adj[i] > eps).sum())
            if current_degree < min_degree:
                num_needed = min_degree - current_degree
                dist = np.linalg.norm(pos - pos[i], axis=1)
                nearest_idxs = np.argsort(dist)[1:]
                added = 0
                for j in nearest_idxs:
                    if adj[i, j] <= eps:
                        adj[i, j] = sim[i, j]
                        adj[j, i] = sim[j, i]
                        added += 1
                    if added >= num_needed:
                        break
        adj = (adj + adj.T) / 2.0
        return torch.tensor(adj, dtype=torch.float32)


# Create test dataset and dataloader
test_dataset = Her2ST(train=False, fold=fold)
test_loader = DataLoader(test_dataset, batch_size=1, num_workers=2,
                         pin_memory=True, shuffle=False)

## 4. Load TITAN and Trained Model Checkpoint

In [ ]:
# Load TITAN for slide-level encoding
local_model_path = "./TITAN"
titan = AutoModel.from_pretrained(
    local_model_path,
    trust_remote_code=True,
    local_files_only=True
)
titan = titan.to(device)

# Initialize VISTA and load trained weights
model = ImgtoGene(n_genes=785)
model.to(device)

state_dict = torch.load(
    f"model/Benchmarking_breasttest_{fold}.ckpt",
    map_location=device
)
model.load_state_dict(state_dict)
print("Checkpoint loaded successfully.")

## 5. Run Prediction

In [ ]:
def predict_and_evaluate(model, test_dataloader, device):
    """
    Run inference on the test set.
    Returns predicted and ground-truth expression matrices as numpy arrays.
    """
    model.eval()
    with torch.no_grad():
        for patch, feature, exp, adj, center, position in test_dataloader:
            patch, feature, exp, adj, center, position = [
                x.to(device) for x in [patch, feature, exp, adj, center, position]]
            exp = exp.squeeze()
            recon_gene = model(patch, feature, adj, center, position)

    adata = recon_gene.squeeze(0).cpu().numpy()
    adata_gt = exp.squeeze(0).cpu().numpy()
    return adata, adata_gt


adata_pred, adata_truth = predict_and_evaluate(model, test_loader, device=device)

## 6. Evaluate and Save Results

In [ ]:
# --- Compute Pearson correlation ---
def cal_Percor_vectorized(true, pred, axis=1):
    """
    Vectorized Pearson correlation.
    axis=1: per-spot across genes. axis=0: per-gene across spots.
    """
    if axis == 0:
        true = true.T
        pred = pred.T
    true_mean = true.mean(axis=1, keepdims=True)
    pred_mean = pred.mean(axis=1, keepdims=True)
    true_centered = true - true_mean
    pred_centered = pred - pred_mean
    numerator = np.sum(true_centered * pred_centered, axis=1)
    denominator = (np.sqrt(np.sum(true_centered**2, axis=1)) *
                   np.sqrt(np.sum(pred_centered**2, axis=1)))
    corr = numerator / (denominator + 1e-8)
    return corr, np.nanmean(corr)


# Spot-level and gene-level PCC
_, corr_mean1 = cal_Percor_vectorized(adata_truth, adata_pred, axis=1)
_, corr_mean2 = cal_Percor_vectorized(adata_truth, adata_pred, axis=0)
print(f"Spot-level PCC: {corr_mean1:.4f}, Gene-level PCC: {corr_mean2:.4f}")

In [ ]:
# --- Build AnnData objects and save ---
final_labels = anndata.AnnData(adata_truth)
final_preds = anndata.AnnData(adata_pred)
final_labels.var_names = test_dataset.gene_list
final_preds.var_names = test_dataset.gene_list
final_labels.obs_names = test_dataset.meta_dict[test_dataset.names[0]].obs_names
final_preds.obs_names = test_dataset.meta_dict[test_dataset.names[0]].obs_names

# Save as h5ad files for downstream analysis
final_labels.write('./experiments/' + test_dataset.names[0] + "_groundtruth.h5ad")
final_preds.write('./experiments/' + test_dataset.names[0] + ".h5ad")
print(f"Saved predictions for {test_dataset.names[0]}")